# P10–P11 — Repeticiones con 5 semillas y evaluación con métricas
## (a) Paso del protocolo y objetivo
**P10:** repetir el baseline y DIET con las semillas 10, 20, 30, 40 y 50 para medir la **variabilidad por semilla**. **P11:** evaluar en el conjunto de prueba con **exactitud, precisión/recobrado macro, F1 macro y exactitud balanceada**, con **intervalos de confianza bootstrap**. Aquí se cubren dos mediciones: el test del **lote 1** (5 semillas; **no cumplió**, F1 0,687) y la **evaluación única del lote 2** (compuerta **G3**; F1 0,9072, recalculado desde las predicciones guardadas). Sustentan el **OE2**.
**Origen:** todo **Real** (cifras agregadas y predicciones sin texto). **No se reevalúa nada**: el test del lote 2 ya se gastó.

## (b) Código del repositorio que lo implementa
`eval_real.py` (lote 1: selecciona en validación y evalúa el test una vez con las 5 semillas), `eval_lote2.py` (lote 2: evaluación única con modelo y umbral congelados; **no se ejecuta aquí**), `crossval_participantes.py` (sensibilidad). Entradas: modelo congelado y partición. Salidas: `logs/v3_real/*.csv`, `logs/v3_real/lote2/`, `logs/avance/eval_lote2_*`.

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("10"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,crossval_participantes.py,Sensibilidad (V1.6),ANÁLISIS DE SENSIBILIDAD — validación cruzada dejando participantes fuera (protocolo V1.2).,Frases reales del lote 1 por participante,logs/avance/cv_participantes_*.md (no cuenta para G3),python scripts/crossval_participantes.py --help,No,Documentado (requiere Rasa)
1,eval_lote2.py,Lote 2 — evaluación ÚNICA (G3),"Lote 2 — evaluación ÚNICA del test (protocolo V1.6, 5.8). Sirve para G3 (criterio F1 macro ≥ 0,75, sin cambios).",Modelo congelado + partición del lote 2,"logs/v3_real/lote2/ (predicciones, F1, IC), logs/avance/eval_lote2_*",python scripts/eval_lote2.py,Sí (import),NO se ejecuta: el cuaderno recalcula desde las predicciones guardadas (etapa 7)
2,eval_real.py,"P08, P10, P11 (lote 1)","Evaluación sobre lenguaje real (protocolo V1.2, P08/P10/P11) — partición v3.",Partición v3 real,"logs/v3_real/ (F1 macro, selección por validación, test UNA vez)",python scripts/eval_real.py # selección + test (Rasa tarda ~1 h 40 min),Sí (import),Documentado (el test del lote 1 ya se gastó)


## (c) Celdas de código
### 1. P10 — Repeticiones con 5 semillas en el test del lote 1 (Real, cifras guardadas)

In [3]:
r = pd.read_csv(BASE / "logs/v3_real/rasa_test.csv")
b = pd.read_csv(BASE / "logs/v3_real/baseline_test.csv")
res1 = leer_json("logs/v3_real/eval_real_resumen.json")
rotulo(ORIGEN_REAL, "test del lote 1 (114 frases reales), 5 semillas — una sola evaluación, ya registrada en test_registro.json")
display(pd.DataFrame({"semilla": r["seed"], "DIET F1 macro": r["f1_macro"].round(4), "DIET exactitud": r["accuracy"].round(4), "SVM F1 macro": b["f1_macro"].round(4), "SVM exactitud": b["accuracy"].round(4)}))
print(f"DIET: media {r['f1_macro'].mean():.4f} · DE entre semillas {r['f1_macro'].std(ddof=1):.4f} · IC95 % guardado {res1['metodos']['rasa']['f1_macro'][1]:.3f}–{res1['metodos']['rasa']['f1_macro'][2]:.3f}")
print(f"SVM : media {b['f1_macro'].mean():.4f} · DE entre semillas {b['f1_macro'].std(ddof=1):.4f} (el SVM es determinista: las 5 corridas son iguales)")
comparar("P10", "F1 macro del lote 1 (DIET, media de 5 semillas)", r["f1_macro"].mean(), 0.6867, "G3 «No cumplida, lote 1»", tol=5e-4)
comparar("P10", "F1 macro del lote 1 frente al JSON", r["f1_macro"].mean(), res1["metodos"]["rasa"]["f1_macro"][0], "logs/v3_real/eval_real_resumen.json", tol=1e-9)
comparar("P10", "DE entre semillas (DIET)", r["f1_macro"].std(ddof=1), res1["metodos"]["rasa"]["sd_f1_semillas"], "logs/v3_real/eval_real_resumen.json", tol=1e-9)
sm = res1["comparacion"]["mcnemar_mayoria"]
print(f"McNemar (mayoría de semillas): solo SVM acierta {sm['solo_svm']}, solo DIET acierta {sm['solo_rasa']}, p = {sm['p']:.3f}")
import importlib
print("Criterio de G3: F1 macro ≥ 0,75 → el lote 1 NO lo cumplió (0,687; el IC95 % por frases [0,570; 0,707] no incluye 0,75). La medición independiente del lote 2 está más abajo.")
DECLARADAS.append("scripts/common.py conserva F1_TARGET = 0,85 (criterio de la V1.1); el criterio vigente de G3 en el Protocolo V1.8 es F1 macro ≥ 0,75.")

▌ORIGEN: Real — test del lote 1 (114 frases reales), 5 semillas — una sola evaluación, ya registrada en test_registro.json


,semilla,DIET F1 macro,DIET exactitud,SVM F1 macro,SVM exactitud
0,10,0.6898,0.7018,0.6676,0.6842
1,20,0.6647,0.6930,0.6676,0.6842
2,30,0.6885,0.7018,0.6676,0.6842
3,40,0.6798,0.7018,0.6676,0.6842
4,50,0.7108,0.7281,0.6676,0.6842


### 2. P11 — Evaluación única del lote 2 (G3): recálculo desde las predicciones guardadas

### Resultados guardados del lote 2 (G3)
**Qué se hace:** a partir de las **predicciones ya guardadas** del modelo congelado (y del SVM) sobre las 267 frases del test del lote 2, se **recalculan** F1 macro, exactitud, cobertura, precisión de lo respondido, abstenciones, los IC bootstrap (por frases y por participantes), el F1 por intención, las intenciones más débiles, la confusión agregada y la prueba de McNemar frente al SVM. Cada cifra se contrasta con la reportada.
**Por qué:** la evaluación única del lote 2 **ya se gastó** (la sección 5.8 del protocolo prohíbe repetirla). Recalcular *desde las predicciones guardadas* permite que cualquiera verifique las cifras **sin volver a evaluar** y sin tocar el modelo.
**Paso del protocolo:** P11 y compuerta **G3** (criterio F1 macro ≥ 0,75, sin cambios).
**Qué NO hace:** no carga ningún modelo, no vuelve a predecir, no cambia el umbral.

In [4]:
def mostrar_comparaciones(etapas=None, titulo="Recalculado frente a reportado"):
    filas = [c for c in COMPARACIONES if etapas is None or c[0] in etapas]
    t = pd.DataFrame(filas, columns=["etapa", "cifra", "recalculado", "reportado", "fuente de lo reportado", "coincide"])
    print(f"{titulo}: {len(t)} cifras · coinciden: {(t['coincide'] == 'Sí').sum()} · NO coinciden: {(t['coincide'] == 'NO').sum()}")
    display(t)


res = leer_json("logs/avance/eval_lote2_resumen.json")
rasa_rep = res["metodos"]["rasa"]
svm_rep = res["metodos"]["svm"]
REP_PROTOCOLO = {"f1_macro": 0.9072, "cobertura": 0.929, "precision_respondida": 0.956, "abstenciones": 19, "n": 267, "participantes": 25, "exactitud": 0.8876}


# cifras que el propio script de evaluación imprimió con el umbral (artefacto declarado)
u = res["umbral_congelado"]
rotulo(ORIGEN_REAL, "lo que imprimió eval_lote2.py en la sección «umbral» (artefacto declarado, NO es la cifra oficial)")
print(f"Sección «umbral» del script: respondidas {u['respondidas']}, abstenciones {u['abstenciones']}, cobertura {u['cobertura']:.1%}, precisión de lo respondido {u['precision_respondida']:.1%}.")
print("Esas cifras cuentan como 'respondidas' las 19 frases que el FallbackClassifier del pipeline ya había convertido en «nlu_fallback»: están declaradas como artefacto en logs/avance/eval_lote2_notas.md.")
DECLARADAS.append("eval_lote2_resumen.json: cobertura 98,9 % y precisión 89,8 % son un artefacto declarado del script (las 19 abstenciones 'nlu_fallback' se contaron como respuestas); las cifras oficiales son 92,9 % y 95,6 %.")

▌ORIGEN: Real — lo que imprimió eval_lote2.py en la sección «umbral» (artefacto declarado, NO es la cifra oficial)
Sección «umbral» del script: respondidas 264, abstenciones 3, cobertura 98.9%, precisión de lo respondido 89.8%.
Esas cifras cuentan como 'respondidas' las 19 frases que el FallbackClassifier del pipeline ya había convertido en «nlu_fallback»: están declaradas como artefacto en logs/avance/eval_lote2_notas.md.


In [5]:
from scipy.stats import binomtest
from sklearn.metrics import f1_score, precision_recall_fscore_support

N_BOOT, SEMILLA = 1000, 42


def f1m(y, p):
    return float(f1_score(np.asarray(y), np.asarray(p), average="macro", zero_division=0))


def ic_bootstrap(y, p, grupos, rng, n=N_BOOT, por_participante=False):
    """(punto, límite inferior, límite superior, media del bootstrap) — misma lógica que scripts/eval_lote2.py"""
    y, p = np.asarray(y), np.asarray(p)
    if por_participante:
        idx = {g: np.where(grupos == g)[0] for g in np.unique(grupos)}
        gs = list(idx)
        vals = [f1m(y[ix], p[ix]) for ix in (np.concatenate([idx[g] for g in rng.choice(gs, len(gs), replace=True)]) for _ in range(n))]
    else:
        vals = [f1m(y[ix], p[ix]) for ix in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return f1m(y, p), float(lo), float(hi), float(np.mean(vals))


if HAY_PRIVADO:
    pred = pd.read_csv(PRIV / "predicciones_lote2_sin_texto.csv", dtype={"confidence": float, "confidence_2": float}, keep_default_na=False, encoding="utf-8")
    y, p, g, ps = pred["intent"].values, pred["predicted"].values, pred["participant_code"].values, pred["predicted_svm"].values
    n = len(y)
    abst = int((p == "nlu_fallback").sum())
    aciertos = int((y == p).sum())
    metricas = {"n": n, "participantes": len(set(g)), "etiquetas en el promedio (esperadas ∪ predichas)": len(set(y) | set(p)),
                "F1 macro": f1m(y, p), "exactitud": aciertos / n, "abstenciones (nlu_fallback)": abst, "cobertura": (n - abst) / n, "precisión de lo respondido": aciertos / (n - abst)}
    f1_54 = f1_score(y, p, labels=sorted(set(y)), average="macro", zero_division=0)
    rotulo(ORIGEN_REAL, f"recalculado desde predicciones guardadas de {n} frases (sin texto)")
    display(pd.DataFrame({"métrica": list(metricas), "recalculado": [round(v, 4) if isinstance(v, float) else v for v in metricas.values()]}))
    print(f"F1 macro promediando solo las 54 intenciones esperadas (sin la etiqueta de abstención): {f1_54:.4f} — declarado en la Nota; el oficial (0,9072) promedia 55 etiquetas y cuenta las abstenciones como error.")
    comparar("G3", "F1 macro (DIET)", metricas["F1 macro"], REP_PROTOCOLO["f1_macro"], "Protocolo V1.8 / eval_lote2_resumen.json")
    comparar("G3", "F1 macro (DIET) frente al JSON del script", metricas["F1 macro"], rasa_rep["f1_macro"][0], "logs/avance/eval_lote2_resumen.json", tol=1e-9)
    comparar("G3", "exactitud", metricas["exactitud"], rasa_rep["accuracy"], "logs/avance/eval_lote2_resumen.json", tol=1e-9)
    comparar("G3", "frases de test", n, REP_PROTOCOLO["n"], "Protocolo V1.8", tol=0)
    comparar("G3", "participantes", metricas["participantes"], REP_PROTOCOLO["participantes"], "Protocolo V1.8", tol=0)
    comparar("G3", "abstenciones (nlu_fallback)", abst, REP_PROTOCOLO["abstenciones"], "Protocolo V1.8 / Nota v11", tol=0)
    comparar("G3", "cobertura", metricas["cobertura"], REP_PROTOCOLO["cobertura"], "Protocolo V1.8 (92,9 %)", tol=5e-4)
    comparar("G3", "precisión de lo respondido", metricas["precisión de lo respondido"], REP_PROTOCOLO["precision_respondida"], "Protocolo V1.8 (95,6 %)", tol=5e-4)
else:
    requiere_privado("predicciones del lote 2 sin texto")
    rotulo(ORIGEN_REAL, "valores GUARDADOS (no recalculados)")
    display(pd.DataFrame({"métrica": ["F1 macro", "exactitud", "abstenciones (nlu_fallback)", "cobertura", "precisión de lo respondido"],
                          "valor guardado": [rasa_rep["f1_macro"][0], rasa_rep["accuracy"], REP_PROTOCOLO["abstenciones"], REP_PROTOCOLO["cobertura"], REP_PROTOCOLO["precision_respondida"]]}))

▌ORIGEN: Real — recalculado desde predicciones guardadas de 267 frases (sin texto)


,métrica,recalculado
0,n,267.0000
1,participantes,25.0000
2,etiquetas en el promedio (esperadas ∪ predichas),55.0000
3,F1 macro,0.9072
4,exactitud,0.8876
5,abstenciones (nlu_fallback),19.0000
6,cobertura,0.9288
7,precisión de lo respondido,0.9556


In [6]:
# ----- Intervalos de confianza bootstrap (1000 remuestreos, semilla 42) -----
if HAY_PRIVADO:
    f1_f = ic_bootstrap(y, p, g, np.random.default_rng(SEMILLA))
    f1_p = ic_bootstrap(y, p, g, np.random.default_rng(SEMILLA), por_participante=True)
    rotulo(ORIGEN_REAL, "IC95 % del F1 macro (DIET) recalculados")
    display(pd.DataFrame([{"remuestreo": "frases", "F1": f1_f[0], "IC95 % inferior": f1_f[1], "IC95 % superior": f1_f[2], "media del bootstrap": f1_f[3]},
                          {"remuestreo": "participantes", "F1": f1_p[0], "IC95 % inferior": f1_p[1], "IC95 % superior": f1_p[2], "media del bootstrap": f1_p[3]}]).round(4))
    comparar("G3", "IC95 % por frases (inferior)", f1_f[1], rasa_rep["f1_macro"][1], "logs/avance/eval_lote2_resumen.json")
    comparar("G3", "IC95 % por frases (superior)", f1_f[2], rasa_rep["f1_macro"][2], "logs/avance/eval_lote2_resumen.json")
    comparar("G3", "IC95 % por participantes (inferior)", f1_p[1], rasa_rep["f1_macro_ic_participantes"][1], "logs/avance/eval_lote2_resumen.json")
    comparar("G3", "IC95 % por participantes (superior)", f1_p[2], rasa_rep["f1_macro_ic_participantes"][2], "logs/avance/eval_lote2_resumen.json")
    comparar("G3", "media del bootstrap por frases", f1_f[3], res["bootstrap"]["media_por_frases"], "logs/avance/eval_lote2_resumen.json")
    comparar("G3", "media del bootstrap por participantes", f1_p[3], res["bootstrap"]["media_por_participantes"], "logs/avance/eval_lote2_resumen.json")
    sobre_umbral_frases = f1_f[1] >= 0.75
    sobre_umbral_part = f1_p[1] >= 0.75
    print(f"¿F1 macro (punto) ≥ 0,75? {f1_f[0] >= 0.75} | ¿límite inferior IC por frases ≥ 0,75? {sobre_umbral_frases} | ¿límite inferior IC por participantes ≥ 0,75? {sobre_umbral_part}")
else:
    requiere_privado("remuestreo bootstrap")
    rotulo(ORIGEN_REAL, "IC95 % GUARDADOS")
    display(pd.DataFrame([{"remuestreo": "frases", "F1": rasa_rep["f1_macro"][0], "inferior": rasa_rep["f1_macro"][1], "superior": rasa_rep["f1_macro"][2]},
                          {"remuestreo": "participantes", "F1": rasa_rep["f1_macro_ic_participantes"][0], "inferior": rasa_rep["f1_macro_ic_participantes"][1], "superior": rasa_rep["f1_macro_ic_participantes"][2]}]).round(4))

▌ORIGEN: Real — IC95 % del F1 macro (DIET) recalculados


,remuestreo,F1,IC95 % inferior,IC95 % superior,media del bootstrap
0,frases,0.9072,0.8626,0.9290,0.8980
1,participantes,0.9072,0.8524,0.9385,0.8997


In [7]:
# ----- F1 por intención, las tres más débiles y confusiones agregadas -----
f1_guardado = pd.read_csv(BASE / "logs/avance/eval_lote2_f1_por_intencion.csv")
if HAY_PRIVADO:
    intenciones = sorted(set(y))
    pre, rec, f1i, sup = precision_recall_fscore_support(y, p, labels=intenciones, zero_division=0)
    por_int = pd.DataFrame({"intent": intenciones, "n": sup, "F1": f1i.round(4)})
    fusion = por_int.merge(f1_guardado[["intent", "f1"]].rename(columns={"f1": "F1 guardado"}), on="intent")
    dif_max = float((fusion["F1"] - fusion["F1 guardado"]).abs().max())
    comparar("G3", "F1 por intención (máxima diferencia absoluta, 54 intenciones)", dif_max, 0.0, "logs/avance/eval_lote2_f1_por_intencion.csv", tol=5e-4)
    rotulo(ORIGEN_REAL, "F1 por intención recalculado (n = frases de test de esa intención)")
    display(por_int.sort_values("F1").reset_index(drop=True))
    debiles = por_int.sort_values(["F1", "intent"]).head(3)
    debiles_guard = f1_guardado.sort_values(["f1", "intent"]).head(3)
    print("Las tres intenciones más débiles:", list(debiles["intent"]), "| F1:", list(debiles["F1"]))
    comparar("G3", "tres intenciones más débiles", ",".join(debiles["intent"]), ",".join(debiles_guard["intent"]), "logs/avance/eval_lote2_f1_por_intencion.csv", tipo="txt")
    # confusiones agregadas (solo etiquetas)
    err = pd.DataFrame({"esperada": y, "predicha": p})
    err = err[err["esperada"] != err["predicha"]]
    top = err.groupby(["esperada", "predicha"]).size().rename("veces").reset_index().sort_values("veces", ascending=False).head(12)
    rotulo(ORIGEN_REAL, "confusión agregada: pares (esperada → predicha) más frecuentes; incluye «nlu_fallback» = abstención")
    display(top.reset_index(drop=True))
    print("Errores totales:", len(err), "de los cuales abstenciones (nlu_fallback):", int((err["predicha"] == "nlu_fallback").sum()), "| confusiones entre intenciones reales:", int((err["predicha"] != "nlu_fallback").sum()))
    cd = res["confusion_par_despedida_agradecimiento"]
    m_dep = {"n": int((y == "despedida").sum()), "predicha_despedida": int(((y == "despedida") & (p == "despedida")).sum()), "predicha_agradecimiento": int(((y == "despedida") & (p == "agradecimiento")).sum())}
    m_agr = {"n": int((y == "agradecimiento").sum()), "predicha_despedida": int(((y == "agradecimiento") & (p == "despedida")).sum()), "predicha_agradecimiento": int(((y == "agradecimiento") & (p == "agradecimiento")).sum())}
    comparar("G3", "par despedida↔agradecimiento", json.dumps([m_dep, m_agr], sort_keys=True), json.dumps([{k: cd["despedida"][k] for k in m_dep}, {k: cd["agradecimiento"][k] for k in m_agr}], sort_keys=True), "logs/avance/eval_lote2_resumen.json", tipo="txt")
    print("Despedida:", m_dep, "| Agradecimiento:", m_agr, "(confusión esperable: «gracias» se usa también para despedirse)")
else:
    requiere_privado("F1 por intención y confusiones")
    rotulo(ORIGEN_REAL, "F1 por intención GUARDADO (las 10 más bajas)")
    display(f1_guardado.sort_values("f1").head(10)[["intent", "n", "f1"]])

▌ORIGEN: Real — F1 por intención recalculado (n = frases de test de esa intención)


,intent,n,F1
0,licencia_funcionamiento_plazo,5,0.5714
1,fuera_de_alcance,15,0.6364
2,denuncia_seguridad_ciudadana,5,0.6667
3,redes_sociales_mpsr,5,0.7500
4,partida_matrimonio,5,0.7500
5,pago_predial_como,4,0.7500
6,despedida,5,0.8000
7,saludo,4,0.8000
8,constancia_domiciliaria,5,0.8000
9,requisitos_mesa_partes,5,0.8889


,esperada,predicha,veces
0,fuera_de_alcance,nlu_fallback,5
1,redes_sociales_mpsr,nlu_fallback,2
2,partida_matrimonio,nlu_fallback,2
3,licencia_funcionamiento_plazo,nlu_fallback,2
4,agradecimiento,despedida,1
5,requisitos_mesa_partes,nlu_fallback,1
6,renovacion_licencia,nlu_fallback,1
7,queja_atencion,presentar_reclamo,1
8,partida_defuncion,nlu_fallback,1
9,pago_predial_como,nlu_fallback,1


In [8]:
# ----- SVM como referencia y McNemar exacto (informativo: G3 se mide solo con DIET) -----
if HAY_PRIVADO:
    f1_svm_f = ic_bootstrap(y, ps, g, np.random.default_rng(SEMILLA))
    f1_svm_p = ic_bootstrap(y, ps, g, np.random.default_rng(SEMILLA), por_participante=True)
    ok_svm, ok_dt = (ps == y), (p == y)
    solo_svm, solo_diet = int((ok_svm & ~ok_dt).sum()), int((~ok_svm & ok_dt).sum())
    p_mc = float(binomtest(min(solo_svm, solo_diet), solo_svm + solo_diet, 0.5).pvalue) if solo_svm + solo_diet else 1.0
    rotulo(ORIGEN_REAL, "DIET frente a SVM (mismas 267 frases)")
    display(pd.DataFrame([{"modelo": "DIET (congelado)", "F1 macro": f1m(y, p), "exactitud": float((p == y).mean())},
                          {"modelo": "SVM (baseline informativo)", "F1 macro": f1_svm_f[0], "exactitud": float((ps == y).mean())}]).round(4))
    print(f"Solo SVM acierta: {solo_svm} · solo DIET acierta: {solo_diet} · p de McNemar exacto = {p_mc:.3f} → no hay evidencia de que uno sea mejor (p > 0,05).")
    comparar("G3", "F1 macro del SVM", f1_svm_f[0], svm_rep["f1_macro"][0], "logs/avance/eval_lote2_resumen.json", tol=1e-9)
    comparar("G3", "McNemar: solo SVM acierta", solo_svm, svm_rep["comparacion_con_rasa"]["solo_svm_acierta"], "logs/avance/eval_lote2_resumen.json", tol=0)
    comparar("G3", "McNemar: solo DIET acierta", solo_diet, svm_rep["comparacion_con_rasa"]["solo_rasa_acierta"], "logs/avance/eval_lote2_resumen.json", tol=0)
    comparar("G3", "McNemar: p exacto", p_mc, svm_rep["comparacion_con_rasa"]["p_mcnemar_exacto"], "logs/avance/eval_lote2_resumen.json", tol=1e-9)
else:
    requiere_privado("McNemar frente al SVM")
    cr = svm_rep["comparacion_con_rasa"]
    rotulo(ORIGEN_REAL, "valores GUARDADOS")
    print(f"SVM F1 macro {svm_rep['f1_macro'][0]:.4f} · solo SVM acierta {cr['solo_svm_acierta']} · solo DIET acierta {cr['solo_rasa_acierta']} · p de McNemar exacto {cr['p_mcnemar_exacto']:.3f}")

▌ORIGEN: Real — DIET frente a SVM (mismas 267 frases)


,modelo,F1 macro,exactitud
0,DIET (congelado),0.9072,0.8876
1,SVM (baseline informativo),0.8925,0.8727


In [9]:
mostrar_comparaciones(etapas=["G3"], titulo="Esta sección — recalculado frente a reportado")
print()
print("Diferencias CONOCIDAS y ya declaradas (se muestran, no se corrigen):")
for d in DECLARADAS:
    print(" -", d)

Esta sección — recalculado frente a reportado: 21 cifras · coinciden: 21 · NO coinciden: 0


,etapa,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,G3,F1 macro (DIET),0.90724,0.9072,Protocolo V1.8 / eval_lote2_resumen.json,Sí
1,G3,F1 macro (DIET) frente al JSON del script,0.90724,0.90724,logs/avance/eval_lote2_resumen.json,Sí
2,G3,exactitud,0.88764,0.88764,logs/avance/eval_lote2_resumen.json,Sí
3,G3,frases de test,267,267,Protocolo V1.8,Sí
4,G3,participantes,25,25,Protocolo V1.8,Sí
5,G3,abstenciones (nlu_fallback),19,19,Protocolo V1.8 / Nota v11,Sí
6,G3,cobertura,0.928839,0.929,"Protocolo V1.8 (92,9 %)",Sí
7,G3,precisión de lo respondido,0.955645,0.956,"Protocolo V1.8 (95,6 %)",Sí
8,G3,IC95 % por frases (inferior),0.862641,0.862641,logs/avance/eval_lote2_resumen.json,Sí
9,G3,IC95 % por frases (superior),0.929019,0.929019,logs/avance/eval_lote2_resumen.json,Sí


### Cómo leer el resultado de esta sección
- **F1 macro = 0,9072** con 267 frases de 25 participantes: supera el criterio 0,75; los límites inferiores de los IC95 % (por frases ≈ 0,86 y por participantes ≈ 0,85) también lo superan. **Pero** el lote 2 usa las mismas 56 situaciones del lote 1, una sola revisora de etiquetas y 4–5 frases por intención: es una medición *independiente en participantes*, **no** prueba de generalización a situaciones nuevas. La validación cruzada del lote 1 dio 0,787–0,799, y el 0,907 es más alto que eso.
- **Cobertura 92,9 % y precisión 95,6 %:** de 267 frases, 19 son abstenciones («nlu_fallback») y 248 son respuestas; 237 de ellas son correctas (237/248 = 95,6 %). La exactitud 0,8876 = 237/267 cuenta las abstenciones como error.
- **Dos advertencias de lectura:** (1) el F1 oficial promedia **55 etiquetas** (las 54 intenciones + «nlu_fallback» como predicción, que vale F1 = 0); sobre las 54 intenciones sería 0,9240. (2) La sección «umbral» impresa por el script (98,9 % / 89,8 %) es un **artefacto declarado**, no la cifra oficial.
- **Intenciones débiles** (`licencia_funcionamiento_plazo`, `fuera_de_alcance`, `denuncia_seguridad_ciudadana`): son las que más se confunden entre sí o se abstienen; `fuera_de_alcance` es una clase «cajón» con 15 frases.
- **DIET vs SVM:** diferencia de 0,015 en F1 y p de McNemar = 0,541: *no* hay evidencia de que DIET supere al SVM en este test.
- En la tabla «recalculado frente a reportado», cualquier «NO» sería una **discrepancia** que se reporta tal cual (no se corrige en silencio).

### 3. Glosario de métricas

In [10]:
rotulo("Definiciones (no son datos del estudio)")
display(pd.DataFrame(meta["glosario"]))

▌ORIGEN: Definiciones (no son datos del estudio)


,término,definición,fórmula,interpretación,limitación
0,F1 macro,Promedio simple del F1 de cada intención (todas pesan igual).,F1_i = 2·P_i·R_i/(P_i+R_i); F1 macro = (1/K)·Σ F1_i,"0 = nada acertado, 1 = perfecto. Criterio de G3: ≥ 0,75.","Con 4–5 frases por intención, cada error mueve mucho el F1 de esa intención. En este proyecto el promedio incluye la etiqueta «nlu_fallback» (abstención) cuando aparece como predicción, y cuenta como error."
1,Exactitud (accuracy),Proporción de frases cuya intención predicha coincide con la esperada.,aciertos / total,"Fácil de leer, pero no distingue entre intenciones frecuentes y raras.",Las abstenciones cuentan como error.
2,Cobertura,Proporción de frases que el asistente RESPONDE (no se abstiene).,(total − abstenciones) / total,Más cobertura = menos «no entendí».,Subir la cobertura bajando el umbral suele bajar la precisión.
3,Precisión de lo respondido,"Entre las frases respondidas, proporción de aciertos.",aciertos / (total − abstenciones),Mide la confiabilidad de lo que el asistente dice.,Es condicional a responder: no premia ni castiga abstenerse.
4,Abstención,"El asistente responde «no entendí» cuando la confianza es < t (0,50) o cuando la diferencia con la segunda intención es < 0,1.","confianza_1 < t o confianza_1 − confianza_2 < 0,1",Prefiere callar a responder mal.,El FallbackClassifier está dentro del pipeline congelado: las predicciones traen la etiqueta «nlu_fallback».
5,IC95 % bootstrap por frases,"Intervalo para el F1 macro obtenido remuestreando FRASES con reemplazo (1000 veces, semilla 42).","percentiles 2,5 y 97,5 de los F1 remuestreados","Si el límite inferior supera 0,75, el criterio se sostiene incluso con el azar del muestreo de frases.",Supone frases independientes; con pocas frases por intención el IC por percentiles queda sesgado hacia abajo (por eso se informa también la media del bootstrap).
6,IC95 % bootstrap por participantes,"Igual, pero remuestreando PARTICIPANTES completos (las frases de una misma persona no son independientes).","ídem, por conglomerados",Es el IC más honesto para generalizar a otras personas.,Solo 25 participantes y 56 situaciones: sigue siendo una muestra pequeña y de conveniencia.
7,McNemar exacto,Compara dos clasificadores sobre las MISMAS frases mirando solo las discordantes (donde uno acierta y el otro no).,"p = 2·P(X ≤ min(b,c)), X ~ Binomial(b+c, 0,5)",p alto = no hay evidencia de que uno sea mejor que el otro.,Con pocas discordantes (24 aquí) tiene poca potencia.
8,Alfa de Cronbach,Consistencia interna de una escala de k ítems.,α = k/(k−1) · (1 − Σ var(ítem_i) / var(suma)),"≥ 0,70 se considera aceptable (criterio de G6).",Con n = 5 es muy inestable; un alfa bajo con n pequeño no permite concluir cuál ítem falla.
9,Kappa de Cohen,Acuerdo entre dos juicios corregido por el azar.,κ = (p_o − p_e)/(1 − p_e),1 = acuerdo perfecto; 0 = el que daría el azar.,"En este proyecto, el «kappa» del lote 2 compara la etiqueta esperada con la revisión de UNA sola revisora: NO es acuerdo entre revisores."


### Cómo leer el resultado de esta sección
Úsalo como consulta rápida. Las limitaciones de la última columna son las que se declaran en la Nota de Desviación y en el protocolo: pocas frases por intención, una sola revisora de etiquetas, y n pequeño en el pre-piloto.

In [11]:
cierre("P10–P11 Repeticiones y métricas")

P10–P11 Repeticiones y métricas — recalculado frente a reportado: 24 cifras · coinciden: 24 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,P10,"F1 macro del lote 1 (DIET, media de 5 semillas)",0.686711,0.6867,"G3 «No cumplida, lote 1»",Sí
1,P10,F1 macro del lote 1 frente al JSON,0.686711,0.686711,logs/v3_real/eval_real_resumen.json,Sí
2,P10,DE entre semillas (DIET),0.016797,0.016797,logs/v3_real/eval_real_resumen.json,Sí
3,G3,F1 macro (DIET),0.90724,0.9072,Protocolo V1.8 / eval_lote2_resumen.json,Sí
4,G3,F1 macro (DIET) frente al JSON del script,0.90724,0.90724,logs/avance/eval_lote2_resumen.json,Sí
5,G3,exactitud,0.88764,0.88764,logs/avance/eval_lote2_resumen.json,Sí
6,G3,frases de test,267,267,Protocolo V1.8,Sí
7,G3,participantes,25,25,Protocolo V1.8,Sí
8,G3,abstenciones (nlu_fallback),19,19,Protocolo V1.8 / Nota v11,Sí
9,G3,cobertura,0.928839,0.929,"Protocolo V1.8 (92,9 %)",Sí


## (d) Cómo leer el resultado
- **Lote 1 (P10):** el F1 macro de DIET en el test real fue **0,687** (DE entre semillas 0,017); el del SVM, 0,668. No cumple el criterio 0,75 y por eso G3 queda registrada como «No cumplida, lote 1».
- **Lote 2 (P11, G3):** F1 macro **0,9072** con IC95 % por frases [0,863; 0,929] y por participantes [0,852; 0,939]; cobertura 92,9 %; precisión de lo respondido 95,6 %; 19 abstenciones. Todas las cifras recalculadas coinciden con las reportadas (tabla «recalculado frente a reportado»).
- Los dos números **no son comparables**: son datos y procedimientos distintos (el lote 1 se usó para elegir y refinar; el lote 2 es una prueba independiente, con el diseño cerrado antes de recoger).

## Limitaciones
- El lote 2 usa las **mismas 56 situaciones** del lote 1 y **una sola revisora** de etiquetas; 4–5 frases por intención; el F1 oficial promedia 55 etiquetas y cuenta las abstenciones como error (sobre 54 intenciones sería 0,9240).
- El 0,907 es más alto que la validación cruzada por participantes del lote 1 (0,787–0,799): no es evidencia de generalización a situaciones nuevas.
- DIET no demuestra ser mejor que el SVM (McNemar p = 0,541 en el lote 2).